In [1]:
import arcpy
arcpy.management.CreateFileGDB(r'C:\Users\colet\Documents\GEOG5890_Capstone\GEOG5890_Capstone_GDB', 'GeoAI_Capstone')

<Result 'C:\\Users\\colet\\Documents\\GEOG5890_Capstone\\GEOG5890_Capstone_GDB\\GeoAI_Capstone.gdb'>

In [3]:
import os
arcpy.env.overwriteOutput = True

In [4]:
crashes = r'C:\Users\colet\Documents\GEOG5890_Capstone\Final_Crash_Dataset\Denver_crashes_2015_2024_solar.shp'
out_gdb = r'C:\Users\colet\Documents\GEOG5890_Capstone\GEOG5890_Capstone_GDB\GeoAI_Capstone.gdb'

In [5]:
projected_crashes = os.path.join(out_gdb, 'Projected_Crashes')
crash_grid = os.path.join(out_gdb, 'Crash_Grid')
grid_counts = os.path.join(out_gdb, 'Crash_Grid_Counts')

In [6]:
new_crs = arcpy.SpatialReference(26913)

arcpy.management.Project(crashes, projected_crashes, new_crs)

<Result 'C:\\Users\\colet\\Documents\\GEOG5890_Capstone\\GEOG5890_Capstone_GDB\\GeoAI_Capstone.gdb\\Projected_Crashes'>

In [9]:
denver_boundary = r'C:\Users\colet\Documents\GEOG5890_Capstone\data\Denver_Municipal_Boundary\Denver_munic_boundary.shp'
den_boundary_proj = os.path.join(out_gdb, 'Denver_Boundary_Projected')

arcpy.management.Project(denver_boundary, den_boundary_proj, new_crs)

<Result 'C:\\Users\\colet\\Documents\\GEOG5890_Capstone\\GEOG5890_Capstone_GDB\\GeoAI_Capstone.gdb\\Denver_Boundary_Projected'>

In [10]:
extent = arcpy.Describe(den_boundary_proj).extent

xmin, ymin = extent.XMin, extent.YMin
xmax, ymax = extent.XMax, extent.YMax

cell_width = 250
cell_height = 250

In [11]:
import math

ncols = math.ceil((xmax - xmin) / cell_width)
nrows = math.ceil((ymax - ymin) / cell_height)

origin = f"{xmin} {ymin}"
y_axis = f"{xmin} {ymin + 10}"

opposite_corner = (
    f"{xmin + ncols * cell_width} "
    f"{ymin + nrows * cell_height}"
)

In [12]:
fishnet_rectangle = os.path.join(out_gdb, 'Denver_Fishnet_Rectangle')

arcpy.management.CreateFishnet(
    out_feature_class = fishnet_rectangle,
    origin_coord = origin,
    y_axis_coord = y_axis,
    cell_width = cell_width,
    cell_height = cell_height,
    number_rows = nrows,
    number_columns = ncols,
    corner_coord = opposite_corner,
    labels = 'NO_LABELS',
    template = '#',
    geometry_type = 'POLYGON')

<Result 'C:\\Users\\colet\\Documents\\GEOG5890_Capstone\\GEOG5890_Capstone_GDB\\GeoAI_Capstone.gdb\\Denver_Fishnet_Rectangle'>

In [14]:
arcpy.analysis.Clip(fishnet_rectangle, den_boundary_proj, crash_grid)

<Result 'C:\\Users\\colet\\Documents\\GEOG5890_Capstone\\GEOG5890_Capstone_GDB\\GeoAI_Capstone.gdb\\Crash_Grid'>

In [15]:
arcpy.management.AddField(crash_grid, 'Grid_ID', 'LONG')
arcpy.management.CalculateField(crash_grid, 'Grid_ID', '!OBJECTID!', 'PYTHON3')

<Result 'C:\\Users\\colet\\Documents\\GEOG5890_Capstone\\GEOG5890_Capstone_GDB\\GeoAI_Capstone.gdb\\Crash_Grid'>

In [16]:
crash_grid_join = os.path.join(out_gdb, 'Crash_Grid_Assignment')

arcpy.analysis.SpatialJoin(target_features = projected_crashes, join_features = crash_grid,
                           out_feature_class = crash_grid_join, join_operation = 'JOIN_ONE_TO_ONE',
                           join_type = 'KEEP_COMMON', match_option = 'WITHIN')

<Result 'C:\\Users\\colet\\Documents\\GEOG5890_Capstone\\GEOG5890_Capstone_GDB\\GeoAI_Capstone.gdb\\Crash_Grid_Assignment'>

In [18]:
year_field = 'year'
month_field = 'month'

arcpy.management.AddField(crash_grid_join, 'YearMonth', 'TEXT', field_length = 10)

<Result 'C:\\Users\\colet\\Documents\\GEOG5890_Capstone\\GEOG5890_Capstone_GDB\\GeoAI_Capstone.gdb\\Crash_Grid_Assignment'>

In [19]:
arcpy.management.CalculateField(crash_grid_join, 'YearMonth',
    f"str(int(!{year_field}!)) + '-' + str(int(!{month_field}!)).zfill(2)", 'PYTHON3')

<Result 'C:\\Users\\colet\\Documents\\GEOG5890_Capstone\\GEOG5890_Capstone_GDB\\GeoAI_Capstone.gdb\\Crash_Grid_Assignment'>

In [20]:
monthly_table = os.path.join(out_gdb, 'Monthly_Crash_Counts')

arcpy.analysis.Statistics(in_table = crash_grid_join, out_table = monthly_table,
                         statistics_fields = [['OBJECTID', 'COUNT']], case_field=['Grid_ID', 'YearMonth'])

print(monthly_table)

C:\Users\colet\Documents\GEOG5890_Capstone\GEOG5890_Capstone_GDB\GeoAI_Capstone.gdb\Monthly_Crash_Counts
